# 🧪 Prueba del pipeline (muestra pequeña)

Este notebook es una **versión de prueba** del pipeline completo. Ejecuta todo el flujo sobre una muestra aleatoria de **500 tweets** para que puedas:

1. Verificar que todo funciona sin errores en tu entorno de Colab.
2. Ver ejemplos reales de cómo clasifica el modelo.
3. Ajustar las categorías de fraude si hace falta **antes** de lanzar el notebook grande.
4. Estimar tiempos reales en tu máquina.

**Tiempo total estimado en Colab Free con T4: ~5-8 minutos**

Cuando todo te convenza, ejecuta el notebook grande sobre el dataset completo.

## 1. Instalación y GPU

In [1]:
!pip install -q bertopic sentence-transformers transformers sentencepiece sacremoses
!pip install -q umap-learn hdbscan
print('Instalación terminada.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 44.4 MB/s eta 0:00:00
Instalación terminada.


In [2]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Sin GPU. Ve a Runtime → Change runtime type → T4 GPU')

CUDA disponible: True
GPU: Tesla T4


## 2. Subir el CSV

In [3]:
from google.colab import files
uploaded = files.upload()  # Sube scam_us_all_posts_2025.csv

Saving scam_us_all_posts_2025.csv to scam_us_all_posts_2025.csv


## 3. Carga, limpieza y muestra

Tomamos una muestra aleatoria de 500 tweets para las pruebas.

In [4]:
import pandas as pd
import re

SAMPLE_SIZE = 500   # puedes bajarlo a 200 si quieres ir aún más rápido
RANDOM_SEED = 42    # para que la muestra sea reproducible

df = pd.read_csv('scam_us_all_posts_2025.csv')
print(f'Tweets originales: {len(df)}')

# Filtros básicos
df = df[df['lang'] == 'en'].copy()
df = df.drop_duplicates(subset='text').reset_index(drop=True)
df = df[df['likely_us'] == True].reset_index(drop=True)

# Limpieza del texto (para topic modeling)
def clean_text_for_topics(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['text'].apply(clean_text_for_topics)
df['n_words'] = df['text_clean'].str.split().str.len()
df = df[df['n_words'] >= 4].reset_index(drop=True)

print(f'Tras limpieza: {len(df)}')

# MUESTRA ALEATORIA
df_sample = df.sample(SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'\n✅ Muestra de prueba: {len(df_sample)} tweets')
df_sample[['text']].head(3)

Tweets originales: 13976
Tras limpieza: 12959

✅ Muestra de prueba: 500 tweets


,text
0,Convince me that we should *STOP* rooting out ...
1,"Sounds like a scam: ""Hey! Hope you're having a..."
2,Minnesota fraud is pathetic! They claim they h...


## 4. BERTopic sobre la muestra

⚠️ Nota: BERTopic necesita bastantes documentos para descubrir patrones claros. Con 500 tweets verás menos tópicos y más tweets en la categoría `-1` (outliers) que con el dataset completo. Es normal y esperado — esto es solo para comprobar que el modelo corre.

Tiempo: ~1-2 minutos.

In [5]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

# Parámetros adaptados a muestra pequeña
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean',
                        cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language='english',
    verbose=True
)

docs = df_sample['text_clean'].tolist()
topics, _ = topic_model.fit_transform(docs)
df_sample['topic'] = topics

print(f'\nTópicos encontrados: {len(set(topics)) - (1 if -1 in topics else 0)}')
topic_model.get_topic_info().head(10)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

2026-04-18 17:15:41,292 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

2026-04-18 17:15:42,406 - BERTopic - Embedding - Completed ✓
2026-04-18 17:15:42,407 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-18 17:15:52,827 - BERTopic - Dimensionality - Completed ✓
2026-04-18 17:15:52,829 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-18 17:15:52,849 - BERTopic - Cluster - Completed ✓
2026-04-18 17:15:52,853 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-18 17:15:52,891 - BERTopic - Representation - Completed ✓



Tópicos encontrados: 8


,Topic,Count,Name,Representation,Representative_Docs
0,-1,192,-1_fraud_amp_trump_scam,"[fraud, amp, trump, scam, just, don, want, fed...",[If SNAP is state run then they should use sta...
1,0,111,0_fraud_waste_abuse_government,"[fraud, waste, abuse, government, need, musk, ...","[He is a FRAUD, What about the fraud, Masks ha..."
2,1,96,1_scam_app_just_trying,"[scam, app, just, trying, don, calls, using, k...","[The scam bots are wild., With all the scam an..."
3,2,22,2_judge_fraud_america_trump,"[judge, fraud, america, trump, ag, called, ny,...",[Trump’s massive $500M civil fraud fine in AG ...
4,3,19,3_doge_waste_waste fraud_security,"[doge, waste, waste fraud, security, fraud, so...","[Waste, fraud, and abuse got hands DOGE Trump ..."
5,4,18,4_election_election fraud_voter_voting,"[election, election fraud, voter, voting, vote...",[Election fraud is real! And nearly always com...
6,5,17,5_care_medicaid_shot_medical,"[care, medicaid, shot, medical, year, covid, y...",[I've heard flu is running rampant and med com...
7,6,13,6_mortgage_housing_mortgage fraud_market,"[mortgage, housing, mortgage fraud, market, ye...",[Part of FEDERAL GOVERNMENT housing fraud scam...
8,7,12,7_minnesota_minnesota fraud_fraud_democrats,"[minnesota, minnesota fraud, fraud, democrats,...",[Democrats are fighting so hard to keep the il...


In [6]:
# Palabras clave de los tópicos encontrados
for tid in topic_model.get_topic_info()['Topic']:
    if tid == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(tid)[:6]]
    print(f'Tópico {tid}: {words}')

Tópico 0: ['fraud', 'waste', 'abuse', 'government', 'need', 'musk']
Tópico 1: ['scam', 'app', 'just', 'trying', 'don', 'calls']
Tópico 2: ['judge', 'fraud', 'america', 'trump', 'ag', 'called']
Tópico 3: ['doge', 'waste', 'waste fraud', 'security', 'fraud', 'social security']
Tópico 4: ['election', 'election fraud', 'voter', 'voting', 'voter fraud', 'fraud']
Tópico 5: ['care', 'medicaid', 'shot', 'medical', 'year', 'covid']
Tópico 6: ['mortgage', 'housing', 'mortgage fraud', 'market', 'years', 'sale']
Tópico 7: ['minnesota', 'minnesota fraud', 'fraud', 'democrats', 'investigated', 'mn']


## 5. Clasificación zero-shot sobre la muestra

Aquí es donde valoras si las categorías que has definido cubren bien el dataset. Si ves muchos tweets en "not related to financial fraud" que sí parecen relevantes, probablemente falte una categoría.

Tiempo: ~2-3 minutos con 500 tweets en T4.

In [7]:
# ⚙️ EDITA LIBREMENTE ESTAS CATEGORÍAS hasta que reflejen bien tus datos

FRAUD_CATEGORIES = [
    'investment or cryptocurrency scam',
    'romance scam',
    'phishing or identity theft',
    'government or IRS impersonation scam',
    'bank or wire fraud',
    'payment app scam (Zelle, Venmo, Cash App)',
    'Ponzi scheme or pyramid scheme',
    'tech support scam',
    'employment or job scam',
    'charity or donation scam',
    'insurance fraud',
    'securities or corporate fraud',
    'tax fraud or evasion',
    'not related to financial fraud'
]

print(f'{len(FRAUD_CATEGORIES)} categorías definidas')

14 categorías definidas


In [8]:
from transformers import pipeline
from tqdm.auto import tqdm

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline('zero-shot-classification',
                      model='facebook/bart-large-mnli', device=device)

BATCH_SIZE = 16
texts = df_sample['text_clean'].tolist()
predictions, scores = [], []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    results = classifier(batch, candidate_labels=FRAUD_CATEGORIES, multi_label=False)
    if isinstance(results, dict):
        results = [results]
    for res in results:
        predictions.append(res['labels'][0])
        scores.append(res['scores'][0])

df_sample['fraud_category'] = predictions
df_sample['fraud_category_score'] = scores

print('\nDistribución de categorías en la muestra:')
print(df_sample['fraud_category'].value_counts())

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  0%|          | 0/32 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Distribución de categorías en la muestra:
fraud_category
employment or job scam                       202
phishing or identity theft                    54
Ponzi scheme or pyramid scheme                42
not related to financial fraud                35
charity or donation scam                      27
tax fraud or evasion                          24
romance scam                                  23
securities or corporate fraud                 22
bank or wire fraud                            21
government or IRS impersonation scam          18
investment or cryptocurrency scam             16
insurance fraud                               10
payment app scam (Zelle, Venmo, Cash App)      6
Name: count, dtype: int64


## 6. 🔍 Inspección manual (¡el paso más importante!)

Aquí es donde ganas intuición sobre si el modelo funciona bien. Revisa ejemplos de cada categoría y piensa:

- ¿Los tweets de cada categoría realmente pertenecen ahí?
- ¿Hay tweets en "not related to financial fraud" que SÍ deberían estar clasificados?
- ¿Falta alguna modalidad de fraude que ves en los datos pero no tienes en la lista?

In [9]:
# Ejemplos por cada categoría
for cat in df_sample['fraud_category'].value_counts().index:
    subset = df_sample[df_sample['fraud_category'] == cat]
    print(f'\n{"="*70}\n📂 {cat}  ({len(subset)} tweets)\n{"="*70}')
    for _, row in subset.head(3).iterrows():
        print(f'  [score={row["fraud_category_score"]:.2f}] {row["text"][:200]}...')


📂 employment or job scam  (202 tweets)
  [score=0.13] Convince me that we should *STOP* rooting out fraud, waste, corruption, and simply piss poor foreign interventions....
  [score=0.17] Minnesota fraud is pathetic! They claim they have been investigated and haven't found any fraud.
The investigators need to be investigated!

Lying bunch of damn scumbags!...
  [score=0.16] No surprise here
Fraud &amp; abuse are rampant.... https://t.co/4GiP36qtNr...

📂 phishing or identity theft  (54 tweets)
  [score=0.49] Sounds like a scam: "Hey! Hope you're having an amazing day. My name is Hakeem, and I’m looking to buy a house in the city you operate in. I’d love your help finding the best property suitable for a h...
  [score=0.24] I predict the Philippines will become the jurisdiction of United States soon very soon.

The direct flights from Philippines will cause deaths to American seniors and increase fraud and identity theft...
  [score=0.22] We told you it was a scam. You're in a Cult. htt

In [10]:
# Tweets con score BAJO → los más dudosos, útil para detectar errores
print('⚠️ Tweets con menor confianza del modelo (revisar):\n')
dudosos = df_sample.nsmallest(10, 'fraud_category_score')[['text', 'fraud_category', 'fraud_category_score']]
for _, row in dudosos.iterrows():
    print(f'[{row["fraud_category"]} | {row["fraud_category_score"]:.2f}]')
    print(f'  {row["text"][:200]}\n')

⚠️ Tweets con menor confianza del modelo (revisar):

[employment or job scam | 0.10]
  I wrote a report back and took my seat  https://t.co/DVo3sw21W7

[romance scam | 0.11]
  Look everyone wants to stop government waist but Elon is breaking the law by talking control from @SpeakerJohnson. Let him get approval for forensic accountants. Firing all the Inspectors General who 

[government or IRS impersonation scam | 0.11]
  America's law making process is broken so the President must use executive orders to get important things done. Democrats give bills that are loaded with junk but have cute names. Inflation reduction 

[employment or job scam | 0.11]
  📢 SPOTLIGHT: 🇦🇪 Verasity and Veraviews make front-page news in Khaleej Times via Zawya, full article on page 3
📰 Verasity’s blockchain platform powers Veraviews to fight ad fraud with proof-of-view te

[employment or job scam | 0.11]
  At what point is war declared on these black robed activists? They are not judges. They are aiding and

## 7. Validación manual (opcional pero MUY recomendado para el TFG)

Coge 50-100 tweets aleatorios, lee cada uno y apunta si estás de acuerdo con la categoría asignada. Esto te da un **accuracy real** que puedes poner en la memoria del TFG.

Ejecutar esta celda te exporta un Excel con los 100 para que los revises a mano.

In [11]:
df_validar = df_sample.sample(100, random_state=123)[
    ['tweet_id', 'text', 'fraud_category', 'fraud_category_score']
].copy()
df_validar['correcto_SI_NO'] = ''   # columna vacía para que la rellenes a mano
df_validar['categoria_correcta'] = ''  # por si quieres apuntar la que sí es

df_validar.to_excel('validacion_manual.xlsx', index=False)
print('✅ Exportado validacion_manual.xlsx')
print('Rellena las columnas "correcto_SI_NO" y "categoria_correcta" y calcula el % de aciertos.')

from google.colab import files
files.download('validacion_manual.xlsx')

✅ Exportado validacion_manual.xlsx
Rellena las columnas "correcto_SI_NO" y "categoria_correcta" y calcula el % de aciertos.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Prueba de traducción al español

Traducimos solo los primeros 50 tweets para verificar calidad. Tiempo: <1 minuto.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = 'Helsinki-NLP/opus-mt-en-es'
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
translator = MarianMTModel.from_pretrained(MODEL_NAME).to(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
translator.eval()

def translate_batch(texts, batch_size=32, max_length=256):
    translations = []
    device = translator.device
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = [str(t) if t else ' ' for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                          truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            out = translator.generate(**inputs, max_length=max_length,
                                     num_beams=2, early_stopping=True)
        translations.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return translations

# Traducir los 50 primeros para inspección
sample_texts = df_sample['text'].head(50).tolist()
sample_es = translate_batch(sample_texts, batch_size=16)

print('\n🔤 Comparación original vs traducción:\n')
for i in range(10):
    print(f'EN: {sample_texts[i][:150]}')
    print(f'ES: {sample_es[i][:150]}')
    print('-' * 70)

## 9. Exportar muestra clasificada (para inspeccionar en Excel)

In [ ]:
# Traducir la muestra completa de 500
all_translations = translate_batch(df_sample['text'].tolist(), batch_size=16)
df_sample['text_es'] = all_translations

cols = ['tweet_id', 'created_at', 'text', 'text_es',
        'fraud_category', 'fraud_category_score', 'topic',
        'username', 'user_location']

df_sample[cols].to_excel('muestra_prueba_500.xlsx', index=False)
files.download('muestra_prueba_500.xlsx')
print('✅ muestra_prueba_500.xlsx listo')

## ✅ Checklist antes de lanzar el notebook grande

Revisa el Excel de la muestra y pregúntate:

- [ ] ¿Las categorías cubren bien los tipos de fraude que aparecen?
- [ ] ¿Hay muchos falsos negativos en "not related to financial fraud"?
- [ ] ¿Los scores de confianza son razonables (>0.3 en general)?
- [ ] ¿La traducción al español es comprensible?
- [ ] ¿Los tiempos de ejecución en tu Colab son aceptables?

Si todo ok → ejecuta el notebook grande (`fraud_tweets_pipeline.ipynb`) sobre los ~14k tweets completos.

Si algo falla → ajusta la lista de `FRAUD_CATEGORIES` (añade/quita/reformula) y vuelve a ejecutar este notebook hasta que quedes contenta.